# RSNA: iki çalışma ile gerçek DICOM cache pilotu

Bu notebook yalnız görüntü ön işleme kontrolüdür; eğitim veya model başarısı ölçümü yapmaz.

1. Kaggle'da **private** notebook oluşturun ve bu dosyayı içe aktarın.
2. Yarışma kurallarını kabul edip **Add Input** ile `rsna-knee-abnormality-detection` yarışmasını ekleyin.
3. **Internet: On**, **Accelerator: None** seçin. Bu küçük pilot için GPU gerekmez.
4. Mac'te mevcut canonical JSONL'den çıkardığınız `PILOT_UIDS = [...]` satırını ilk kod hücresine yapıştırın. Rapor, rapor parçaları ve etiket dosyalarını buraya yüklemeyin.
5. Hücreleri sırayla çalıştırın. Başarı halinde yalnız seçtiğiniz çalışmaların cache'i ZIP'e konur.

Kaynak sürümü sabittir: `b4954c06e51d9a6f7c25383fe9cc1b1955b12d63`.
LM Studio değişiklikleriniz Mac'te kalır; bu notebook LLM çağırmaz.
Notebook burada DICOM biçimindeki test dosyalarıyla doğrulanmıştır. Gerçek Kaggle MR girdileriyle çalıştırma henüz doğrulanmamıştır.


In [ ]:
RUN_ID = "pilot-20261003-021558"
PILOT_UIDS = []  # Mac'te üretilen PILOT_UIDS satırıyla değiştirin.

import re
if not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_ID):
    raise ValueError("RUN_ID yalnız harf, rakam, tire ve alt çizgi içerebilir.")
if not isinstance(PILOT_UIDS, list) or not 1 <= len(PILOT_UIDS) <= 20:
    raise ValueError("Mac'teki canonical etiketlerden 1–20 StudyInstanceUID girin.")
if any(not isinstance(uid, str) or not re.fullmatch(r"[0-9]+(?:\.[0-9]+)*", uid) for uid in PILOT_UIDS):
    raise ValueError("Geçerli DICOM StudyInstanceUID değerleri gerekiyor.")
if len(set(PILOT_UIDS)) != len(PILOT_UIDS):
    raise ValueError("UID listesinde tekrar var.")
print({"run_id": RUN_ID, "n_requested": len(PILOT_UIDS)})


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys
import os
import json

COMMIT = "b4954c06e51d9a6f7c25383fe9cc1b1955b12d63"
WORK = Path("/kaggle/working")
REPO = Path("/tmp/rsna-cache-source")
if not REPO.exists():
    subprocess.run(["git", "clone", "--no-checkout", "https://github.com/bulsahkecici/RSNA_Knee_Abnormality_Detection.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", COMMIT], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if actual != COMMIT:
    raise RuntimeError("Kaynak sürümü doğrulanamadı.")

# Projenin tüm eğitim bağımlılıklarını kurmayız; cache modülü bunları kullanmaz.
packages = {"numpy": "numpy", "PIL": "Pillow", "pydicom": "pydicom==3.0.1"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
from rsna_knee.data.cache_build import build_cache, manifest_ready
from rsna_knee.hashing import sha256_file
import numpy as np
print({"source_commit": actual, "python": sys.version.split()[0], "numpy": np.__version__})


In [ ]:
INPUT = Path("/kaggle/input")
candidates = [INPUT / "rsna-knee-abnormality-detection", INPUT / "competitions" / "rsna-knee-abnormality-detection"]
candidates += sorted(INPUT.glob("*/rsna-knee-abnormality-detection"))
roots = sorted({p for p in candidates if (p / "train_series.csv").is_file()})
if len(roots) != 1:
    raise RuntimeError(f"Yarışma girdisi bulunamadı veya belirsiz: {roots}")
DATA_ROOT = roots[0]
DICOM_ROOT = next((DATA_ROOT / name for name in ("train_series", "train") if (DATA_ROOT / name).is_dir()), None)
if DICOM_ROOT is None:
    raise RuntimeError(f"DICOM dizini bulunamadı: {[p.name for p in DATA_ROOT.iterdir()]}")
DEST = WORK / RUN_ID / "cache"
print({"input": str(DATA_ROOT), "dicom_root": str(DICOM_ROOT), "cache": str(DEST)})


In [ ]:
manifest = build_cache(
    series_csv=DATA_ROOT / "train_series.csv",
    dicom_root=DICOM_ROOT,
    dest=DEST,
    study_ids=PILOT_UIDS,
    limit=len(PILOT_UIDS),
    size=224,
    n_centers=3,
    resume=True,
)
print(json.dumps({
    "run_id": RUN_ID,
    "n_requested": manifest["n_requested"],
    "n_studies": manifest["n_studies"],
    "n_quarantine": manifest["n_quarantine"],
    "quarantine": manifest["quarantine"],
}, indent=2))

ready, reason = manifest_ready(DEST / "manifest.json")
if not ready or manifest["quarantine"] or set(manifest["studies"]) != set(PILOT_UIDS):
    raise RuntimeError(f"Cache tamamlanmadı: {reason}. Üstteki quarantine nedenlerini inceleyin; ZIP üretilmedi.")
for uid in PILOT_UIDS:
    meta = manifest["studies"][uid]
    arr = np.load(DEST / f"{uid}.npy", mmap_mode="r", allow_pickle=False)
    if arr.shape != (3, 3, 3, 224, 224) or arr.dtype != np.uint8:
        raise RuntimeError(f"Beklenmeyen tensor: {uid}: {arr.shape}, {arr.dtype}")
    if meta.get("synthetic") is not False or not any(meta.get("slot_mask", [])):
        raise RuntimeError(f"Gerçek ve okunabilen görüntü şartı sağlanmadı: {uid}")
    if sha256_file(DEST / f"{uid}.npy") != meta["sha256"]:
        raise RuntimeError(f"SHA256 uyuşmazlığı: {uid}")
    print({"uid": uid, "shape": list(arr.shape), "dtype": str(arr.dtype), "slot_mask": meta["slot_mask"]})


## Codec hatası olursa

`codec_or_decode` varsa bu komutu ayrı bir hücrede çalıştırın ve cache hücresini yeniden çalıştırın:

```python
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg", "python-gdcm"], check=True)
```

Paket kurulumu oturumu yeniden başlatmanızı isterse yeniden başlatın ve hücreleri baştan çalıştırın. Başka hata türlerinde codec kurmak çözüm değildir; quarantine nedenini inceleyin. Eksik bir görüntü düzlemi `slot_mask=0` olarak temsil edilir; en az bir okunabilen düzlem zorunludur.


In [ ]:
import zipfile
from IPython.display import FileLink, display

# Önceki hücre başarısızsa eski manifest üzerinden çıktı üretmeyin.
ready, reason = manifest_ready(DEST / "manifest.json")
saved = json.loads((DEST / "manifest.json").read_text())
if not ready or saved.get("quarantine") or set(saved.get("studies", {})) != set(PILOT_UIDS):
    raise RuntimeError(f"ZIP ön kontrolü başarısız: {reason}")
bundle_meta = {
    "run_id": RUN_ID, "source_commit": COMMIT, "requested_uids": PILOT_UIDS,
    "purpose": "DICOM preprocessing pilot; no training or performance claim",
    "manifest_sha256": sha256_file(DEST / "manifest.json"),
}
(DEST.parent / "cache_bundle.json").write_text(json.dumps(bundle_meta, indent=2) + "\n")
ZIP = WORK / f"rsna_cache_{RUN_ID}.zip"
tmp = ZIP.with_suffix(".zip.part")
with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(DEST / "manifest.json", "cache/manifest.json")
    for uid in PILOT_UIDS:
        for suffix in (".npy", ".json"):
            archive.write(DEST / f"{uid}{suffix}", f"cache/{uid}{suffix}")
    archive.write(DEST.parent / "cache_bundle.json", "cache_bundle.json")
with zipfile.ZipFile(tmp) as archive:
    if archive.testzip() is not None:
        raise RuntimeError("ZIP bütünlük kontrolü başarısız.")
tmp.replace(ZIP)
print(json.dumps({"stage": "CACHE_READY", "run_id": RUN_ID, "n_studies": len(PILOT_UIDS), "n_quarantine": 0, "zip": str(ZIP), "zip_bytes": ZIP.stat().st_size, "sha256": sha256_file(ZIP)}, indent=2))
display(FileLink(str(ZIP)))


ZIP'i `/kaggle/working` çıktılarından veya son hücredeki bağlantıdan indirin. Mevcut `data/cache` klasörünü üzerine yazmayın: önce ayrı bir klasöre açın ve `cache/manifest.json` dosyasını kontrol edin. Bu iki çalışmalık çıktı eğitim için yeterli veri veya güvenilir validasyon ölçümü sağlamaz. Sonraki eğitim pilotunda etiket kapsamı, eğitim/validasyon ayrımı ve gold eval dışlama ayrıca kontrol edilmelidir.
